In [ ]:
import os
import sys
print(sys.path) 

# sys.path.insert(0, "/mnt/nas/pritish/CMUVERA_IR_ref")
sys.path.insert(0, "/mnt/dgx/pritish/CMUVERA_IR_ref")
sys.path.insert(0, "/raid/infolab/pritish/CMUVERA_IR_ref")
print(sys.path)

['', '/mnt/home/pritish/.local/lib/python3.11/site-packages', '/mnt/nas/pritish', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python311.zip', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/lib-dynload', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages', '/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages/setuptools/_vendor']
['/raid/infolab/pritish/CMUVERA_IR_ref', '/mnt/dgx/pritish/CMUVERA_IR_ref', '', '/mnt/home/pritish/.local/lib/python3.11/site-packages', '/mnt/nas/pritish', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python311.zip', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/lib-dynload', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages', '/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT', '/mnt/nas/pritish/conda/.c

In [ ]:
os.chdir("/mnt/dgx/data/pritish/CMUVERA_IR_ref")

In [3]:
import torch
import os
import numpy as np
from omegaconf import OmegaConf
import time
import logging
from src.utils import set_seed,set_seed_from_checkpoint, load, save
from tqdm import tqdm
import random
from numpy.random import default_rng
import submodlib
import pickle

from src.dataloader import get_dataloader
from src.embedder import ColBERTEmbedder

from src.utils import partial_chamfer_sim_batched_with_rerank
import torch.multiprocessing as mp

# 1) Make sure the env var is set *inside* Python too
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

# 2) Turn on PyTorch’s deterministic‐only mode
torch.use_deterministic_algorithms(True)

from loguru import logger

/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages/beir/util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [4]:
%load_ext autoreload
%autoreload 1

%aimport src.endtoend
%aimport make_plots

In [5]:
config = {
    "k": 15,
    "method": "sml",
    "bucket_size": 0,
    "dataset_name": "hotpotqa",
    "mv_type": 'colbertv2-plaid',
    "mode": "disk"
}

In [6]:
def make_config(config, optim="lazy"): # lazy, ltl, naive
    sys.argv = ["", f"k={config['k']}", f"method={config['method']}",
                f"baseline.bucket_size={config['bucket_size']}",
                f"data.dataset_name={config['dataset_name']}",
                f"embedder.mv_type={config['mv_type']}",
                f"submodlib.optimizer={optim}",
                f"embedder.mode={config.get('mode', 'mem')}"
    ]
    print(sys.argv)
    
    cli_conf = OmegaConf.from_cli()
    file_config = OmegaConf.load("configs/config.yaml")

    conf = OmegaConf.merge(file_config, cli_conf)

    return conf

In [7]:
conf = make_config(config)

['', 'k=15', 'method=sml', 'baseline.bucket_size=0', 'data.dataset_name=hotpotqa', 'embedder.mv_type=colbertv2-plaid', 'submodlib.optimizer=lazy', 'embedder.mode=disk']


In [8]:
def get_method(config):
    name = config.method
    if name == "v0":
        return src.endtoend.GreedyBaseline_v0(config)
    elif name=="sml":
        return src.endtoend.GreedyBaseline_submodlib(config)
    else:
        raise ValueError(f"Unknown method: {name}")

In [9]:
retriever = get_method(conf)
retriever.run()

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5233329/5233329 [00:22<00:00, 232020.31it/s]


<function ColBERTEmbedder.embed_full_dataset.<locals>.<lambda> at 0x7fb1527963e0>
./experiments/hotpotqa/BERT


/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()



#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: Were Scott Derrickson and Ed Wood of the same nationality?, 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  2020,  3660, 18928,  3385,  1998,  3968,  3536,  1997,
         1996,  2168, 10662,  1029,   102,   103,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')



/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()


./experiments/hotpotqa/BERT/corpus/compressed_128/batch_0.0.pkl
./experiments/hotpotqa/BERT/corpus/compressed_128/batch_1.0.pkl
./experiments/hotpotqa/BERT/corpus/compressed_128/batch_2.0.pkl
./experiments/hotpotqa/BERT/corpus/compressed_128/batch_3.0.pkl
./experiments/hotpotqa/BERT/corpus/compressed_128/batch_4.0.pkl
./experiments/hotpotqa/BERT/corpus/compressed_128/batch_5.0.pkl
./experiments/hotpotqa/BERT/corpus/compressed_128/batch_6.0.pkl
./experiments/hotpotqa/BERT/corpus/compressed_128/batch_7.0.pkl
./experiments/hotpotqa/BERT/corpus/compressed_128/batch_8.0.pkl
./experiments/hotpotqa/BERT/corpus/compressed_128/batch_9.0.pkl
./experiments/hotpotqa/BERT/corpus/compressed_128/batch_10.0.pkl
./experiments/hotpotqa/BERT/corpus/compressed_128/batch_11.0.pkl
./experiments/hotpotqa/BERT/corpus/compressed_128/batch_12.0.pkl
./experiments/hotpotqa/BERT/corpus/compressed_128/batch_13.0.pkl
./experiments/hotpotqa/BERT/corpus/compressed_128/batch_14.0.pkl
./experiments/hotpotqa/BERT/corpus/

Corpus: 0it [00:00, ?it/s]/mnt/dgx/data/pritish/CMUVERA_IR_ref/src/embedder.py:288: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cemb = torch.load(self.embedding_path(f"com

KeyboardInterrupt: 

In [ ]:
# conf = make_config(config, optim="naive")

['', 'k=15', 'method=sml', 'baseline.bucket_size=0', 'data.dataset_name=scifact', 'embedder.mv_type=colbertv2-plaid', 'submodlib.optimizer=naive', 'embedder.mode=mem']


In [ ]:
# retriever = get_method(conf)
# retriever.run()

In [ ]:
conf = make_config(config, optim="ltl")

['', 'k=15', 'method=sml', 'baseline.bucket_size=0', 'data.dataset_name=scifact', 'embedder.mv_type=colbertv2-plaid', 'submodlib.optimizer=ltl', 'embedder.mode=mem']


In [ ]:
retriever = get_method(conf)
retriever.run()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5183/5183 [00:00<00:00, 140499.57it/s]



#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: 0-dimensional biomaterials show inductive properties., 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  1014,  1011,  8789, 16012,  8585, 14482,  2015,  2265,
        27427, 14194,  6024,  5144,  1012,   102,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

./experiments/scifact/BERT/corpus/compressed_128/batch_0.0.pkl
0 1 loaded
0 2 loaded


Corpus: 1it [00:05,  5.04s/it]
Query: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 300/300 [00:01<00:00, 157.30it/s]15]


(tensor([[4564, 2716, 2625, 3054, 2022, 2022],
         [ 899, 4814, 3044, 3044, 3044, 3044],
         [2862,  604, 1063,  138,  138,  138],
         ...,
         [2122,  692, 1819, 1624, 1624, 1624],
         [2128, 2152, 4850, 4850, 4850, 4850],
         [2030, 2290,  842,   92, 1608, 1608]]),
 tensor([[11.6709, 13.9376, 15.4270, 16.5174, 17.6283, 17.6283],
         [17.1472, 19.2077, 20.5443, 20.5443, 20.5443, 20.5443],
         [13.0809, 16.4909, 18.8917, 20.9276, 20.9276, 20.9276],
         ...,
         [13.5287, 17.6820, 20.2283, 24.5195, 24.5195, 24.5195],
         [18.9021, 22.8279, 24.2486, 24.2486, 24.2486, 24.2486],
         [14.6383, 17.0231, 18.7446, 19.9932, 21.1217, 21.1217]]))

In [ ]:
# os.makedirs("plots/html",exist_ok=True)
# os.makedirs("plots/images",exist_ok=True)   
# # Load the data
# datasets = ["scifact"]
# b_sizes_aug = [10,25,50,100,200]
# k_values=[15]
# max_k = 15
# for data in datasets:
#     make_plots.plot_k_vs_metric(data,max_k,b_sizes_aug)
#     make_plots.plot_b_vs_metric(data,k_values,b_sizes_aug)

In [ ]:
with open("./pickles/results/greedy_base_0_128_k15_scifact_bf.pkl", "rb") as f:
    results = pickle.load(f)

# with open("./pickles/results/greedy_submodlib_NaiveGreedy_k15_scifact_bf_k15.pkl", "rb") as f:
#     results_naive = pickle.load(f)

with open("./pickles/results/greedy_submodlib_LazyGreedy_k15_scifact_bf_k15.pkl", "rb") as f:
    results_lazy = pickle.load(f)

with open("./pickles/results/greedy_submodlib_LazierThanLazyGreedy_k15_scifact_bf_k15.pkl", "rb") as f:
    results_ltl = pickle.load(f)

In [ ]:
results_lazy[0][0], results_lazy[1][0]

(tensor([2716, 1858, 2660, 2022, 2022, 2022]),
 tensor([11.8889, 15.3268, 17.4311, 18.8824, 18.8824, 18.8824]))

In [ ]:
results_ltl[0][0], results_ltl[1][0]

(tensor([4564, 2716, 2625, 3054, 2022, 2022]),
 tensor([11.6709, 13.9376, 15.4270, 16.5174, 17.6283, 17.6283]))

In [ ]:
results[0]

[(2716, 11.888933181762695),
 (1858, 15.326838493347168),
 (2660, 17.4311466217041),
 (2022, 18.882389068603516),
 (1853, 19.841398239135742),
 (190, 20.44447898864746),
 (1321, 20.73671531677246),
 (916, 20.99953842163086),
 (4404, 21.165863037109375),
 (1421, 21.295223236083984),
 (1632, 21.38498306274414),
 (2967, 21.44413185119629),
 (1864, 21.495826721191406),
 (4658, 21.539615631103516),
 (2495, 21.570064544677734)]

In [ ]:
# naive, lazy and greedy baseline should all give the same results. ltl may differ.
for item_ind in range(len(results)):
    # Check if naive and greedy baseline results match
    greedy_eles = [x[0] for x in results[item_ind]]
    greedy_scores = [x[1] for x in results[item_ind]]

    # res_naive = results_naive[0][item_ind].tolist()
    # res_naive_scores = results_naive[1][item_ind].tolist()

    res_lazy = results_lazy[0][item_ind].tolist()
    res_lazy_scores = results_lazy[1][item_ind].tolist()

    # assert set(res_naive).issubset(set(greedy_eles)), f"Mismatch at index {item_ind}: {greedy_eles} != {res_naive}"
    # assert set(res_naive_scores).issubset(set(greedy_scores)), f"Mismatch at index {item_ind}: {greedy_scores} != {res_naive_scores}"
    
    # Check if naive and lazy baseline results match
    assert set(res_lazy).issubset(set(greedy_eles)), f"Mismatch at index {item_ind}: {greedy_eles} != {res_lazy}"
    # assert greedy_scores == res_lazy_scores, f"Mismatch at index {item_ind}: {greedy_scores} != {res_lazy_scores}"